# Dividend Intelligence Platform (DIP)

A personal-finance research prototype for turning market data into dividend-investing insights. This notebook implements the proposal's first MVP slice: **data ingestion, historical analysis, opportunity screening, income planning, and portfolio visualization**.

## Product Workflow

1. Collect price, dividend, and company data from Yahoo Finance.
2. Normalize the results into a per-ticker analytics table.
3. Compare opportunities using transparent historical indicators.
4. Estimate the shares and capital needed for a quarterly income target.
5. Review watchlist trends and identify follow-up research.

The prototype uses a configurable watchlist and is designed to be extended. It does not establish dividend safety, intrinsic value, or future returns. Historical distributions can change, and the data provider may be incomplete or delayed. Verify figures with company filings before investing.

## MVP Scope and Roadmap

**Implemented here:** historical prices and distributions, descriptive income metrics, heuristic screening, target-income scenarios, and charts.

**Next product phases:** dividend sustainability and risk models, forecasting and alerts, portfolio simulation, persistent watchlists, a curated database, an authenticated FastAPI service, and a React dashboard deployed to Azure. The notebook is the analytics prototype, not the production platform architecture.

In [10]:
%pip install --quiet yfinance plotly

Note: you may need to restart the kernel to use updated packages.


In [11]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 1. Configure the Research Universe

Start with a small watchlist that can be changed without editing the pipeline. The symbols below preserve the original set, with the intended Realty Income ticker corrected to `GTY`.

In [12]:
WATCHLIST = ["PFE", "GTY", "KO", "MO", "DOW", "BKH"]
HISTORY_PERIOD = "5y"
QUARTERLY_INCOME_TARGET = 1200.00

histories = {}
price_frames = []
data_issues = []

for symbol in WATCHLIST:
    try:
        history = yf.Ticker(symbol).history(
            period=HISTORY_PERIOD,
            auto_adjust=False,
            actions=True,
        )
        if history.empty:
            data_issues.append(f"{symbol}: no history returned")
            continue

        frame = history[["Close", "Dividends"]].copy()
        frame.index = pd.to_datetime(frame.index).tz_localize(None)
        frame.index.name = "Date"
        frame = frame.reset_index()
        frame["Symbol"] = symbol
        histories[symbol] = frame
        price_frames.append(frame[["Symbol", "Date", "Close", "Dividends"]])
    except Exception as error:
        data_issues.append(f"{symbol}: {type(error).__name__}: {error}")

prices = (
    pd.concat(price_frames, ignore_index=True)
    if price_frames
    else pd.DataFrame(columns=["Symbol", "Date", "Close", "Dividends"])
)

print(f"Loaded {len(histories)} of {len(WATCHLIST)} symbols ({HISTORY_PERIOD} requested).")
if data_issues:
    print("Data retrieval issues:")
    print("\n".join(f"- {issue}" for issue in data_issues))
prices.tail()

Loaded 6 of 6 symbols (5y requested).


,Symbol,Date,Close,Dividends
7525,BKH,2026-10-12,76.32,0.00
7526,BKH,2026-10-13,77.24,0.00
7527,BKH,2026-10-14,76.43,0.00
7528,BKH,2026-10-15,75.87,0.00
7529,BKH,2026-10-16,76.63,0.00


## 2. Build Curated Historical Metrics

The pipeline keeps daily closes and distribution events in one tidy table. For each company, it calculates the trailing 12-month cash distribution per share, an explicitly historical comparison with the prior 12 months, price range, and realized volatility. A trailing yield is a backward-looking ratio, not a promised yield.

In [13]:
metric_rows = []
as_of = prices["Date"].max() if not prices.empty else pd.Timestamp.today().normalize()

for symbol, history in histories.items():
    history = history.sort_values("Date").copy()
    closes = history["Close"].dropna()
    distributions = history.loc[history["Dividends"] > 0, ["Date", "Dividends"]]
    trailing_start = as_of - pd.Timedelta(days=365)
    prior_start = as_of - pd.Timedelta(days=730)
    trailing_dividend = distributions.loc[
        distributions["Date"] > trailing_start, "Dividends"
    ].sum()
    prior_dividend = distributions.loc[
        (distributions["Date"] > prior_start)
        & (distributions["Date"] <= trailing_start),
        "Dividends",
    ].sum()
    dividend_growth = (
        trailing_dividend / prior_dividend - 1
        if prior_dividend > 0
        else np.nan
    )
    yearly_distributions = distributions.groupby(
        distributions["Date"].dt.year
    )["Dividends"].sum()
    observed_years = min(5, max(1, as_of.year - history["Date"].min().year + 1))
    positive_distribution_years = int((yearly_distributions > 0).tail(5).sum())
    daily_returns = closes.pct_change().dropna()
    drawdown = closes / closes.cummax() - 1

    metric_rows.append(
        {
            "Symbol": symbol,
            "Latest Price": closes.iloc[-1] if not closes.empty else np.nan,
            "Trailing 12M Dividend / Share": trailing_dividend,
            "Trailing Yield": trailing_dividend / closes.iloc[-1]
            if not closes.empty and closes.iloc[-1] > 0
            else np.nan,
            "Dividend Growth vs Prior 12M": dividend_growth,
            "Payments in Trailing 12M": int(
                (distributions["Date"] > trailing_start).sum()
            ),
            "Dividend-Paying Years (5Y)": positive_distribution_years,
            "Observed Years": observed_years,
            "52W Low": closes.loc[history["Date"] >= as_of - pd.Timedelta(days=365)].min(),
            "52W High": closes.loc[history["Date"] >= as_of - pd.Timedelta(days=365)].max(),
            "Annualized Volatility": daily_returns.std() * np.sqrt(252),
            "Max Drawdown (Price)": drawdown.min(),
        }
    )

analytics = pd.DataFrame(metric_rows).set_index("Symbol")
analytics

,Latest Price,Trailing 12M Dividend / Share,Trailing Yield,Dividend Growth vs Prior 12M,Payments in Trailing 12M,Dividend-Paying Years (5Y),Observed Years,52W Low,52W High,Annualized Volatility,Max Drawdown (Price)
Symbol,,,,,,,,,,,
PFE,29.58,1.72,0.06,0.01,4,5,5,23.67,29.58,0.26,-0.65
GTY,28.38,1.94,0.07,0.03,4,5,5,26.85,36.61,0.21,-0.30
KO,93.90,2.10,0.02,0.04,4,5,5,67.54,93.90,0.17,-0.21
MO,75.11,4.29,0.06,0.04,4,5,5,54.72,75.11,0.22,-0.31
DOW,29.93,1.40,0.05,-0.43,4,5,5,20.95,41.87,0.34,-0.71
BKH,76.63,2.78,0.04,0.04,4,5,5,62.40,77.24,0.23,-0.40


## 3. Screen and Compare Opportunities

This ranking is a transparent research heuristic, not a buy recommendation or a dividend-safety model. It combines capped trailing yield, the share of observed years with distributions, positive trailing dividend growth, and lower historical price volatility. Scores are comparable only within this watchlist and depend on the selected history window.

In [14]:
yield_component = (analytics["Trailing Yield"].clip(lower=0, upper=0.05) / 0.05) * 25
consistency_component = (
    analytics["Dividend-Paying Years (5Y)"]
    / analytics["Observed Years"].replace(0, np.nan)
).clip(lower=0, upper=1) * 50
growth_component = (
    analytics["Dividend Growth vs Prior 12M"].fillna(0).clip(lower=0, upper=0.10)
    / 0.10
) * 15
volatility_component = (
    1 - analytics["Annualized Volatility"].fillna(0.50) / 0.50
).clip(lower=0, upper=1) * 10

analytics["Opportunity Score"] = (
    yield_component + consistency_component + growth_component + volatility_component
).round(1)

screen = analytics.sort_values("Opportunity Score", ascending=False)
screen[
    [
        "Opportunity Score",
        "Latest Price",
        "Trailing 12M Dividend / Share",
        "Trailing Yield",
        "Dividend Growth vs Prior 12M",
        "Payments in Trailing 12M",
        "Dividend-Paying Years (5Y)",
        "Annualized Volatility",
    ]
]

,Opportunity Score,Latest Price,Trailing 12M Dividend / Share,Trailing Yield,Dividend Growth vs Prior 12M,Payments in Trailing 12M,Dividend-Paying Years (5Y),Annualized Volatility
Symbol,,,,,,,,
MO,86.80,75.11,4.29,0.06,0.04,4,5,0.22
GTY,85.60,28.38,1.94,0.07,0.03,4,5,0.21
PFE,80.80,29.58,1.72,0.06,0.01,4,5,0.26
BKH,79.60,76.63,2.78,0.04,0.04,4,5,0.23
DOW,76.60,29.93,1.40,0.05,-0.43,4,5,0.34
KO,74.10,93.90,2.10,0.02,0.04,4,5,0.17


In [15]:
annual_dividend = analytics["Trailing 12M Dividend / Share"]
latest_price = analytics["Latest Price"]

income_scenarios = analytics[
    ["Latest Price", "Trailing 12M Dividend / Share", "Trailing Yield"]
].copy()
income_scenarios["Shares to Target"] = np.ceil(
    (QUARTERLY_INCOME_TARGET * 4) / annual_dividend.replace(0, np.nan)
)
income_scenarios["Estimated Capital"] = (
    income_scenarios["Shares to Target"] * latest_price
)
income_scenarios["Estimated Avg Quarterly Income"] = (
    income_scenarios["Shares to Target"] * annual_dividend / 4
)

print(f"Quarterly income target: ${QUARTERLY_INCOME_TARGET:,.2f}")
income_scenarios.sort_values("Estimated Capital")

Quarterly income target: $1,200.00


,Latest Price,Trailing 12M Dividend / Share,Trailing Yield,Shares to Target,Estimated Capital,Estimated Avg Quarterly Income
Symbol,,,,,,
GTY,28.38,1.94,0.07,"2,475.00","70,240.50","1,200.38"
PFE,29.58,1.72,0.06,"2,791.00","82,557.78","1,200.13"
MO,75.11,4.29,0.06,"1,119.00","84,048.09","1,200.13"
DOW,29.93,1.40,0.05,"3,429.00","102,629.97","1,200.15"
BKH,76.63,2.78,0.04,"1,724.00","132,110.12","1,200.33"
KO,93.90,2.10,0.02,"2,286.00","214,655.40","1,200.15"


## 4. Plan Toward an Income Goal

The calculator uses trailing 12-month distributions as an annual run rate. It converts that figure to an average quarter for planning; it does not guarantee the amount or timing of any future payment. The “shares to target” column treats each ticker as a standalone scenario, not a recommendation to concentrate in one company.

In [16]:
shares_held = pd.Series(0.0, index=analytics.index, name="Shares Held")
# Edit the values below to match your current positions, for example: shares_held.loc["KO"] = 25.

portfolio = analytics[["Latest Price", "Trailing 12M Dividend / Share"]].copy()
portfolio["Shares Held"] = shares_held
portfolio["Estimated Market Value"] = (
    portfolio["Shares Held"] * portfolio["Latest Price"]
)
portfolio["Estimated Annual Income"] = (
    portfolio["Shares Held"] * portfolio["Trailing 12M Dividend / Share"]
)
portfolio["Estimated Avg Quarterly Income"] = portfolio["Estimated Annual Income"] / 4

portfolio_totals = pd.DataFrame(
    {
        "Estimated Market Value": [portfolio["Estimated Market Value"].sum()],
        "Estimated Annual Income": [portfolio["Estimated Annual Income"].sum()],
        "Estimated Avg Quarterly Income": [
            portfolio["Estimated Avg Quarterly Income"].sum()
        ],
    },
    index=["Watchlist Portfolio"],
)

portfolio, portfolio_totals

(        Latest Price  Trailing 12M Dividend / Share  Shares Held  \
 Symbol                                                             
 PFE            29.58                           1.72         0.00   
 GTY            28.38                           1.94         0.00   
 KO             93.90                           2.10         0.00   
 MO             75.11                           4.29         0.00   
 DOW            29.93                           1.40         0.00   
 BKH            76.63                           2.78         0.00   
 
         Estimated Market Value  Estimated Annual Income  \
 Symbol                                                    
 PFE                       0.00                     0.00   
 GTY                       0.00                     0.00   
 KO                        0.00                     0.00   
 MO                        0.00                     0.00   
 DOW                       0.00                     0.00   
 BKH                      

## 5. Watchlist and Portfolio Dashboard

Set current share counts in the next cell to estimate portfolio income from the same trailing distribution data. The estimates assume the recent 12-month run rate persists; they do not account for taxes, reinvestment, fees, or changes to future distributions.

In [17]:
indexed_prices = prices.sort_values(["Symbol", "Date"]).copy()
indexed_prices["Indexed Price"] = indexed_prices.groupby("Symbol")["Close"].transform(
    lambda series: series / series.iloc[0] * 100
)

price_chart = px.line(
    indexed_prices,
    x="Date",
    y="Indexed Price",
    color="Symbol",
    title="Watchlist Price Trends (First Observation = 100)",
    labels={"Indexed Price": "Indexed price", "Date": "Date"},
)
price_chart.update_layout(hovermode="x unified", legend_title_text="Ticker")
price_chart.show()

risk_return_chart = px.scatter(
    analytics.reset_index(),
    x="Annualized Volatility",
    y="Trailing Yield",
    color="Opportunity Score",
    hover_name="Symbol",
    title="Historical Yield and Price Volatility",
    labels={
        "Annualized Volatility": "Annualized daily price volatility",
        "Trailing Yield": "Trailing 12-month yield",
        "Opportunity Score": "Heuristic score",
    },
)
risk_return_chart.show()

## Product Roadmap

- **MVP prototype:** watchlist ingestion, historical dividend and price metrics, transparent screening, income scenarios, and visual analysis (this notebook).
- **Analytics expansion:** sustainability and risk research, valuation context, portfolio simulation, alert rules, and forward-looking scenarios with clearly stated assumptions.
- **Production data platform:** scheduled ingestion with provider terms and rate limits respected, raw and curated storage, data-quality checks, and repeatable ETL/ELT workflows.
- **Application platform:** authenticated FastAPI services, a React experience, persistent watchlists, monitoring, automated tests, CI/CD, and Azure deployment.
- **Business development:** begin with a free tool, then evaluate premium analytics and advisor-focused services against user demand.

A production scheduler should be introduced with a reliable data provider, storage, logging, and retry controls; repeated notebook calls are not a substitute for that pipeline.